# Meta-Learning with MAML on Meta-World ML1

This notebook demonstrates the implementation of Model-Agnostic Meta-Learning (MAML) on the `ML1` benchmark from the `metaworld` library. 

The `ML1` benchmark tests an agent's ability to quickly adapt to variations of a single task. In this example, we use the `reach-v2` task, where the robot arm must reach different goal positions. MAML will learn a set of initial parameters that can be effectively fine-tuned for any new goal position with only a few gradient steps.

## 1. Setup and Imports

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import gymnasium as gym
import metaworld
from metaworld.envs import sawyer_reach_v3
from metaworld.wrappers import RandomTaskSelectWrapper
import matplotlib.pyplot as plt
from functools import partial

import sys
sys.path.append('..')

from meta_rl.algorithms.maml import MAML, ActorCritic
from meta_rl.utils.data_utils import collect_trajectories, process_trajectories

jax.config.update('jax_platform_name', 'cpu')

## 2. MAML Training on Meta-World ML1

In [ ]:
# Hyperparameters
META_ITERATIONS = 50 # Reduced for faster demo
META_BATCH_SIZE = 10
TRAJECTORIES_PER_TASK = 5
MAX_STEPS_PER_TRAJECTORY = 150
INNER_LR = 0.01
META_LR = 0.001

# Setup environment and MAML agent
env_name = 'reach-v3'
ml1_benchmark = metaworld.ML1(env_name, seed=0)

base_env = sawyer_reach_v3.SawyerReachEnvV3(render_mode='rgb_array')
ml1_env = RandomTaskSelectWrapper(base_env, ml1_benchmark.train_tasks)

obs_dim = ml1_env.observation_space.shape[0]
action_dim = ml1_env.action_space.shape[0]

maml = MAML(action_dim=action_dim, obs_dim=obs_dim, inner_lr=INNER_LR, meta_lr=META_LR)

rng = jax.random.PRNGKey(0)
rng, key = jax.random.split(rng)
meta_params, opt_state = maml.init_params(key)

policy_fn = maml.network.apply
losses = []

print(f"Starting MAML training on Meta-World ML1 ({env_name})...")
for meta_iter in range(META_ITERATIONS):
    support_batches, query_batches = [], []

    for _ in range(META_BATCH_SIZE):
        # The RandomTaskSelectWrapper will automatically sample and set a new task
        # on each call to env.reset() inside collect_trajectories.
        rng, key = jax.random.split(rng)
        support_traj = collect_trajectories(ml1_env, meta_params, policy_fn, TRAJECTORIES_PER_TASK, MAX_STEPS_PER_TRAJECTORY, key)
        support_batch = process_trajectories(support_traj, policy_fn, meta_params)
        support_batches.append(support_batch)
        
        rng, key = jax.random.split(rng)
        query_traj = collect_trajectories(ml1_env, meta_params, policy_fn, TRAJECTORIES_PER_TASK, MAX_STEPS_PER_TRAJECTORY, key)
        query_batch = process_trajectories(query_traj, policy_fn, meta_params)
        query_batches.append(query_batch)

    stacked_support = jax.tree_util.tree_map(lambda *x: jnp.stack(x), *support_batches)
    stacked_query = jax.tree_util.tree_map(lambda *x: jnp.stack(x), *query_batches)
    meta_batch = (stacked_support, stacked_query)
    
    meta_params, opt_state, loss = maml.outer_update(meta_params, opt_state, meta_batch)
    losses.append(loss)
    
    if (meta_iter + 1) % 10 == 0:
        print(f"Meta-iteration {meta_iter + 1}/{META_ITERATIONS}, Loss: {loss:.4f}")

print("Training finished.")

## 3. Visualizing Results and Meta-Testing

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.title('MAML Meta-Loss on ML1')
plt.xlabel('Meta-Iteration')
plt.ylabel('Loss')
plt.grid(True)
plt.show()